In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

# Make src/ importable from notebook context
PROJECT_ROOT = Path().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DIR_DB_GOLD, ML_TARGET_COLUMN

# ── Structural columns — never treated as ML features ──────────────────────
STRUCTURAL_COLS = {"silver_id", "quarter", "year", "period_enddate", "BedrijfstakkenBranchesSBI2008"}

print(f"Project root : {PROJECT_ROOT}")
print(f"Gold DB      : {DIR_DB_GOLD}")
print(f"Target column: {ML_TARGET_COLUMN}")

Project root : C:\Users\gebruiker\Desktop\UWV Project\Code\eaisi-uwv
Gold DB      : C:\Users\gebruiker\Desktop\UWV Project\Code\eaisi-uwv\data\3_gold\gold_data.db
Target column: Ziekteverzuimpercentage_1


In [2]:
from src.ml_engineering.ml_1_data_extraction import DataExtractor

extractor = DataExtractor(db_path=DIR_DB_GOLD, table_name="master_data_ml_preprocessed")
df_raw = extractor.extract(target_column=ML_TARGET_COLUMN, feature_groups=None)  # discovery: all columns

print(f"Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

⚙️ SBI mode=all-industry (BedrijfskenmerkenSBI2008_T001081) | 120 quarterly rows
✅ Extraction complete | mode=discovery | sbi=all-industry | features=398 | rows=120


Shape: 120 rows × 402 columns


,period_enddate,year,quarter,Ziekteverzuimpercentage_1,trend_index,AdemtGeenStoffenIn_24_MBG0095,AdemtGeenStoffenIn_24_MOG0095,AdemtGeenStoffenIn_24_MW00000,AdemtStoffenIn_20_MBG0095,AdemtStoffenIn_20_MOG0095,...,GewerkteUren_5_A045285,GewerkteUren_5_A045286,WerkzamePersonenSeizoengecorrigeerd_2_A045285,WerkzamePersonenSeizoengecorrigeerd_2_A045286,WerkzamePersonenSeizoengecorrigeerd_9_A045285,WerkzamePersonenSeizoengecorrigeerd_9_A045286,WerkzamePersonen_1_A045285,WerkzamePersonen_1_A045286,WerkzamePersonen_7_A045285,WerkzamePersonen_7_A045286
0,1996-03-31 00:00:00.000000,1996.0,1.0,5.5,1.0,4.0,3.7,3.9,5.5,4.5,...,216.866667,44.733333,589.733333,102.600000,0.800000,-0.066667,582.933333,102.000000,13.2,0.733333
1,1996-06-30 00:00:00.000000,1996.0,2.0,4.6,2.0,4.0,3.7,3.9,5.5,4.5,...,202.733333,46.600000,594.066667,102.600000,4.333333,0.066667,596.466667,102.733333,15.2,0.266667
2,1996-09-30 00:00:00.000000,1996.0,3.0,4.0,3.0,4.0,3.7,3.9,5.5,4.5,...,198.866667,43.866667,597.866667,102.733333,3.800000,0.266667,604.133333,102.866667,14.8,0.400000


In [3]:
SBI_FILTER_PREFIX = "BedrijfskenmerkenSBI2008_"

structural    = [c for c in df_raw.columns if c in STRUCTURAL_COLS]
target_col    = [ML_TARGET_COLUMN]
sector_filter = [c for c in df_raw.columns if c.startswith(SBI_FILTER_PREFIX)]

# Feature columns: everything else — exclude structural, target, and sector-filter OHE
feature_cols  = [
    c for c in df_raw.columns
    if c not in STRUCTURAL_COLS
    and c != ML_TARGET_COLUMN
    and not c.startswith(SBI_FILTER_PREFIX)
]

summary = pd.DataFrame({
    "Bucket":  ["structural", "target", "sector_filter (OHE)", "feature"],
    "Count":   [len(structural), len(target_col), len(sector_filter), len(feature_cols)],
    "Examples": [
        ", ".join(structural[:3]),
        ML_TARGET_COLUMN,
        ", ".join(sector_filter[:3]),
        ", ".join(feature_cols[:3]),
    ],
})
print(summary.to_string(index=False))
print(f"\nTotal ML features (excl. structural + target + sector_filter): {len(feature_cols)}")

example_col = sector_filter[0] if sector_filter else "BedrijfskenmerkenSBI2008_XXX"
print(f"\nSector filter columns ({len(sector_filter)}) — use to restrict analysis to one SBI sector:")
print(f"  e.g. df_raw[df_raw[{example_col!r}] == 1]")

             Bucket  Count                                                                  Examples
         structural      3                                             period_enddate, year, quarter
             target      1                                                 Ziekteverzuimpercentage_1
sector_filter (OHE)      0                                                                          
            feature    398 trend_index, AdemtGeenStoffenIn_24_MBG0095, AdemtGeenStoffenIn_24_MOG0095

Total ML features (excl. structural + target + sector_filter): 398

Sector filter columns (0) — use to restrict analysis to one SBI sector:
  e.g. df_raw[df_raw['BedrijfskenmerkenSBI2008_XXX'] == 1]
